In [24]:
import pandas as pd
from collections import defaultdict

In [ ]:
matches = pd.read_csv("../data/processed/all_matches.csv")
matches.head()

,Season,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,...,BVCH,BVCD,BVCA,CLCH,CLCD,CLCA,LBCH,LBCD,LBCA,MatchDateTime
0,21-22,E0,13/08/2021,20:00,Brentford,Arsenal,2,0,H,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2021-08-13 20:00:00
1,21-22,E0,14/08/2021,12:30,Man United,Leeds,5,1,H,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2021-08-14 12:30:00
2,21-22,E0,14/08/2021,15:00,Burnley,Brighton,1,2,A,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2021-08-14 15:00:00
3,21-22,E0,14/08/2021,15:00,Chelsea,Crystal Palace,3,0,H,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2021-08-14 15:00:00
4,21-22,E0,14/08/2021,15:00,Everton,Southampton,3,1,H,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2021-08-14 15:00:00


In [19]:
def calculate_team_stats(last5, team):
    stats = {
        "GoalsLast5": 0,
        "PointsLast5": 0,
        "GoalsAgainstLast5": 0,
        "ShotsForLast5": 0,
        "ShotsOnTargetLast5": 0,
        "ShotsAgainstLast5": 0,
        "ShotsOnTargetAgainstLast5": 0,
    }

    if not last5:
        return stats

    goals = []
    goals_against = []
    shots_for = []
    shots_ot_for = []
    shots_against = []
    shots_ot_against = []
    points = 0

    for match in last5:
        if match["HomeTeam"] == team:
            goals.append(match["FTHG"])
            goals_against.append(match["FTAG"])

            shots_for.append(match["HS"])
            shots_ot_for.append(match["HST"])

            shots_against.append(match["AS"])
            shots_ot_against.append(match["AST"])

            if match["FTR"] == "H":
                points += 3
            elif match["FTR"] == "D":
                points += 1

        else:  # team was away
            goals.append(match["FTAG"])
            goals_against.append(match["FTHG"])

            shots_for.append(match["AS"])
            shots_ot_for.append(match["AST"])

            shots_against.append(match["HS"])
            shots_ot_against.append(match["HST"])

            if match["FTR"] == "A":
                points += 3
            elif match["FTR"] == "D":
                points += 1

    stats["GoalsLast5"] = sum(goals) / len(goals)
    stats["PointsLast5"] = points
    stats["GoalsAgainstLast5"] = sum(goals_against) / len(goals_against)
    stats["ShotsForLast5"] = sum(shots_for) / len(shots_for)
    stats["ShotsOnTargetLast5"] = sum(shots_ot_for) / len(shots_ot_for)
    stats["ShotsAgainstLast5"] = sum(shots_against) / len(shots_against)
    stats["ShotsOnTargetAgainstLast5"] = sum(shots_ot_against) / len(shots_ot_against)

    return stats

In [20]:

# Store feature columns automatically
features = defaultdict(list)
team_history = defaultdict(list)

for _, current_match in matches.iterrows():

    home_team = current_match["HomeTeam"]
    away_team = current_match["AwayTeam"]

    home_last5 = team_history[home_team][-5:]
    away_last5 = team_history[away_team][-5:]

    home_stats = calculate_team_stats(home_last5, home_team)
    away_stats = calculate_team_stats(away_last5, away_team)

    for key, value in home_stats.items():
        features[f"Home{key}"].append(value)

    for key, value in away_stats.items():
        features[f"Away{key}"].append(value)

    # Update history AFTER calculating features
    team_history[home_team].append(current_match)
    team_history[away_team].append(current_match)

# Add all features to dataframe
for col, values in features.items():
    matches[col] = values

In [21]:
matches[
    ["HomeTeam",
    "AwayTeam",
    "MatchDateTime",
    "FTHG",
    "FTAG",
    "FTR",
    "HomeGoalsLast5",
    "AwayGoalsLast5",
    "HomePointsLast5",
    "AwayPointsLast5",
    "HomeGoalsAgainstLast5",
    "AwayGoalsAgainstLast5",
    "HomeShotsForLast5",
    "HomeShotsOnTargetLast5",
    "HomeShotsAgainstLast5",
    "HomeShotsOnTargetAgainstLast5",
    "AwayShotsForLast5",
    "AwayShotsOnTargetLast5",
    "AwayShotsAgainstLast5",
    "AwayShotsOnTargetAgainstLast5"
    ]
][1000:1010]

,HomeTeam,AwayTeam,MatchDateTime,FTHG,FTAG,FTR,HomeGoalsLast5,AwayGoalsLast5,HomePointsLast5,AwayPointsLast5,HomeGoalsAgainstLast5,AwayGoalsAgainstLast5,HomeShotsForLast5,HomeShotsOnTargetLast5,HomeShotsAgainstLast5,HomeShotsOnTargetAgainstLast5,AwayShotsForLast5,AwayShotsOnTargetLast5,AwayShotsAgainstLast5,AwayShotsOnTargetAgainstLast5
1000,Fulham,Aston Villa,2024-02-17 15:00:00,1,2,A,1.4,2.0,8,7,1.0,1.4,15.2,5.4,17.6,3.6,16.8,7.6,12.0,4.4
1001,Newcastle,Bournemouth,2024-02-17 15:00:00,2,2,D,2.8,0.8,7,2,2.8,2.4,11.4,4.8,19.4,8.6,15.6,3.0,10.0,5.6
1002,Nott'm Forest,West Ham,2024-02-17 15:00:00,2,0,H,1.6,0.6,4,3,2.0,2.4,10.0,3.4,11.2,4.0,11.6,2.8,17.8,7.0
1003,Tottenham,Wolves,2024-02-17 15:00:00,1,2,A,2.4,2.0,11,7,1.6,1.6,14.4,5.8,12.4,3.8,13.4,5.6,13.2,4.8
1004,Man City,Chelsea,2024-02-17 17:30:00,1,1,D,2.6,2.0,15,9,0.8,2.2,20.6,7.4,7.6,2.8,12.4,5.0,16.8,6.8
1005,Sheffield United,Brighton,2024-02-18 14:00:00,0,5,A,1.4,1.0,4,5,2.6,1.4,10.2,4.2,16.0,5.6,12.2,4.6,11.0,4.8
1006,Luton,Man United,2024-02-18 16:30:00,1,2,A,2.4,2.4,5,10,2.2,1.6,15.6,6.4,12.0,5.2,13.8,5.0,17.0,5.6
1007,Everton,Crystal Palace,2024-02-19 20:00:00,1,1,D,0.4,1.6,3,6,1.4,3.0,12.0,2.4,16.2,5.2,11.4,4.8,13.2,5.2
1008,Man City,Brentford,2024-02-20 19:30:00,1,0,H,2.4,1.8,13,6,1.0,2.4,23.2,7.6,8.6,3.6,10.6,5.0,17.6,7.2
1009,Liverpool,Luton,2024-02-21 19:30:00,4,1,H,3.2,2.2,12,5,1.2,2.0,18.4,7.8,10.8,4.2,17.0,6.0,13.8,5.4


In [23]:
matches.to_csv("../data/processed/features_basic.csv", index=False)